# 09 — The optimizer landscape: a field guide

**What you'll learn**

- The whole optimizer zoo on two axes: **what** is tuned × **how** the search works.
- The weights track: `BootstrapFinetune`, and the BetterTogether result (paper ↔ code map).
- Structure-level moves: `Ensemble`, plus dynamic demos (`KNNFewShot`) and SIMBA.
- **The chooser** — which optimizer to reach for, given your data, budget, and constraints.

*Time: ~20 min. Cost: $0–0.05 (one optional ensemble evaluation; the rest is reading). Cached reruns are free.*

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import dspy
from dotenv import load_dotenv              # pip install -r requirements.txt if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead
# STRONG_MODEL = "openrouter/moonshotai/kimi-k2.6"
# STRONG_MODEL = "openrouter/openai/gpt-4o"

lm = dspy.LM(MODEL, max_tokens=1000)        # cache=True by default: reruns are ~free
dspy.configure(lm=lm)

### Phoenix observability (optional)

Run the next cell to stream this notebook's LM calls into a local [Phoenix](https://phoenix.arize.com) trace UI — or skip it; nothing below depends on tracing. It finds or starts the Phoenix server, prints the link, and names the session by date (`dspy-lab-YYYY-MM-DD`) so different days never mix. Notebook 11 tells the full story.

In [ ]:
# optional: Phoenix tracing (see notebook 11)
import obs

obs.enable_phoenix()

### MLflow observability (optional)

Run the next cell to record this notebook's LM calls with MLflow instead (or as well): logs land in a repo-local store that is safe against remnants and reruns, browsable any time with `obs.mlflow_ui()`. Equally optional; equally explained in notebook 11.

In [ ]:
# optional: MLflow tracing (see notebook 11)
import obs

obs.enable_mlflow()

Boilerplate — same code as notebooks 03 and 04, used only by the optional ensemble experiment below. Skim and run.

In [ ]:
# === boilerplate: aldervane micro-world — same code as notebook 03 ===
import json, math, re
from collections import Counter
from pathlib import Path

CORPUS = json.loads(Path("data/corpus.json").read_text())

STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "did", "do", "does", "each", "every",
    "for", "from", "he", "her", "his", "in", "is", "it", "its", "of", "on",
    "or", "she", "that", "the", "this", "to", "was", "were", "what", "when",
    "where", "which", "who", "whom", "whose", "with",
}

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    # naive stemming: "captains" -> "captain", "records" -> "record"
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
             for w in words]
    return [w for w in words if w not in STOPWORDS]

DOC_TOKENS = {p["id"]: set(tokenize(p["title"] + " " + p["text"])) for p in CORPUS}
IDF = Counter()
for tokens in DOC_TOKENS.values():
    IDF.update(tokens)
IDF = {t: math.log(len(CORPUS) / df) for t, df in IDF.items()}

def search(query, k=2):
    """Return the k corpus passages that best match the query (keyword + IDF)."""
    q = set(tokenize(query))
    ranked = sorted(
        CORPUS,
        key=lambda p: (-sum(IDF.get(t, 0.0) for t in q & DOC_TOKENS[p["id"]]), p["id"]),
    )
    return ranked[:k]

def as_context(passages):
    return [f"{p['title']}: {p['text']}" for p in passages]

class HopQA(dspy.Module):
    """Two-hop retrieval QA over the Aldervane corpus."""

    def __init__(self, k=2):
        super().__init__()
        self.k = k
        self.hop1 = dspy.Predict("question -> search_query")
        self.hop2 = dspy.Predict("question, context: list[str] -> search_query")
        self.answer = dspy.ChainOfThought("question, context: list[str] -> answer")

    def forward(self, question):
        q1 = self.hop1(question=question).search_query
        first = search(q1, k=self.k)
        q2 = self.hop2(question=question, context=as_context(first)).search_query
        second = search(q2, k=self.k)
        passages = {p["id"]: p for p in first + second}      # merge + dedupe
        pred = self.answer(question=question, context=as_context(list(passages.values())))
        return dspy.Prediction(
            answer=pred.answer,
            reasoning=pred.reasoning,
            queries=[q1, q2],
            retrieved_ids=sorted(passages),
        )

In [ ]:
# === boilerplate: data + metric — same code as notebook 04 ===
QA = json.loads(Path("data/qa.json").read_text())

def to_examples(rows):
    return [dspy.Example(**row).with_inputs("question") for row in rows]

train, dev, test = (to_examples(QA[s]) for s in ("train", "dev", "test"))

def norm(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def em_metric(example, pred, trace=None):
    """1.0 if the predicted answer exactly matches the gold answer (after normalization)."""
    return norm(pred.answer) == norm(example.answer)

## The map

Every optimizer you have met — and every one you haven't — answers two questions: *which parameters does it write?* and *how does it search?* Lay the axes out and the zoo becomes a grid:

```
                                    HOW IT SEARCHES
 WHAT IT TUNES     copy      bootstrap     random      Bayesian     reflective     gradient
                   labels    + filter      search      surrogate    evolution      descent
 ---------------  --------  ------------  ----------  -----------  -------------  ----------
 demos            Labeled    Bootstrap     Bootstrap
                  FewShot    FewShot       RS
                  KNNFewShot*
 instructions                              COPRO**     MIPROv2      GEPA
 demos + instr.                                        MIPROv2      SIMBA
 LM weights                                                                        Bootstrap
                                                                                   Finetune
 program set      Ensemble (no search: you supply the candidates)
 ---------------  --------------------------------------------------------------------------
  * KNNFewShot picks demos per input, at runtime      ** COPRO: LLM-guided hill climb
```

Notebooks 05–07 walked the demos row and the instructions row left to right — each step trading more compute for a smarter search. Two regions remain: the **weights** row and the **program set** row.

## The weights track

Everything so far changed *text around a frozen model*. If you control the model itself (open weights, or a provider's finetuning API), the bootstrap trick from notebook 05 has a second life: those metric-filtered teacher traces are, seen from another angle, **a finetuning dataset**. Same traces; demos put them in the prompt, finetuning puts them in the weights.

```python
# The pattern (not runnable here: OpenRouter does not expose finetuning; you need
# a local model or a finetune-capable provider configured for the student LM):
optimizer = dspy.BootstrapFinetune(metric=qa_metric)
finetuned_program = optimizer.compile(HopQA(), trainset=train)
```

Why bother, when demos worked? Latency and cost — learned behavior moves out of the prompt (zero extra tokens per call), and small finetuned models can match much larger prompted ones on narrow tasks.

The natural follow-up — *prompts or weights?* — turns out to be a false choice. **BetterTogether** (*Fine-Tuning and Prompt Optimization: Two Great Steps that Work Better Together*, Soylu et al., 2024, [arXiv:2407.10930](https://arxiv.org/abs/2407.10930)):

| Paper concept | Plain English | Where it lives in code |
|---|---|---|
| Alternating prompt- and weight-optimization | Optimize prompts, finetune on the resulting traces, re-optimize prompts on the new weights... beats either alone | `dspy.BetterTogether(metric=...)` with a strategy string, e.g. `strategy="p -> w"` or `"p -> w -> p"` |
| The prompt step ("p") | Bootstrap-family compile | internally `BootstrapFewShotWithRandomSearch` |
| The weight step ("w") | Traces → finetuning data → updated LM | internally `BootstrapFinetune` |

(In dspy 3.3.0 the strategy grammar has grown past the paper — strings like `"mipro -> gepa -> mipro"` compose *named* optimizers into a schedule. The paper's alternation insight, generalized into an API.)

## The program-set track: ensembles

Cheapest trick in the book, straight from classical ML: run several *diverse* programs, vote. `dspy.Ensemble` is barely an optimizer — `compile(programs)` takes the candidate list you built and returns a module that runs them and reduces the outputs (note the signature break: no student, no trainset — the "search" already happened when you made the candidates).

Diversity below comes free: three `LabeledFewShot` compiles with different `k` (zero LM calls to build — only the evaluation costs anything):

In [ ]:
def vote(predictions):
    """Majority vote on the normalized answer; ties break toward the first program."""
    from collections import Counter
    counts = Counter(norm(p.answer) for p in predictions)
    winner, _ = counts.most_common(1)[0]
    return next(p for p in predictions if norm(p.answer) == winner)

candidates = [dspy.LabeledFewShot(k=k).compile(HopQA(), trainset=train) for k in (2, 4, 8)]
ensemble = dspy.Ensemble(reduce_fn=vote).compile(candidates)

evaluate = dspy.Evaluate(devset=dev, metric=em_metric, num_threads=8, display_progress=True)
evaluate(ensemble).score

> **What you should see:** an ensemble score at or a little above the best single candidate — classic variance reduction (bagging, with programs for base learners). Each dev question now costs 3 programs × 3 LM calls, the standing price of ensembles. (`dspy.majority` is the built-in reduce for simple single-answer programs; we wrote `vote` to use our `norm`.)

## Two briefs to complete the zoo

**SIMBA** — *Stochastic Introspective Mini-Batch Ascent*. `dspy.SIMBA(metric=..., bsize=32, num_candidates=6, max_steps=8)`. Each step: sample a minibatch, find where candidate programs *disagree most* (highest score variance — the informative examples), then either self-reflect a new rule from the contrast between good and bad trajectories or promote a winning demo; keep what measures better. Kin to GEPA (LLM reads execution feedback) but organized around minibatch ascent. Per this course's convention we note plainly: **no standalone paper — a DSPy-native method** (docs + source are the reference).

**KNNFewShot** — the demos row's runtime twist. `dspy.KNNFewShot(k, trainset=trainset, vectorizer=dspy.Embedder(...))` embeds the trainset; at **each call** it retrieves the k nearest training examples to the *current input* and uses those as demos (then bootstrap-processed). Fixed demos answer "what is this task?"; KNN demos answer "what does *this input's neighborhood* look like?" — worth it when your inputs cluster into regimes. Signature break to remember: the trainset lives in the **constructor**; `compile(student)` takes no trainset. The vectorizer can be a hosted embedding model (`dspy.Embedder("openai/text-embedding-3-small")`) or any callable mapping `list[str]` to vectors.

## The chooser

```
                          START: do you have a metric and ~20+ examples?
                                  |                        |
                                 no                       yes
                                  |                        |
                   write them first (nb 04);      how many labeled examples?
                   meanwhile: zero-shot +                  |
                   Refine/BestOfN (nb 08)     ~10-50 ------+------ 200+
                                  |               |                  |
                                  |     BootstrapFewShot(+RS)   budget & needs?
                                  |     (nb 05) - cheap, robust      |
                                  |               |            +----+---------+------------+
                                  |     want 0-shot prompts,   |    |         |            |
                                  |     shorter latency, or    | MIPROv2   rich textual  own the
                                  |     rubric-y tasks?        | medium/   feedback      weights?
                                  |               |            | heavy     available?    |
                                  |             COPRO /        | (nb 06)     |        Bootstrap
                                  |             MIPROv2 light  |           GEPA       Finetune /
                                  |             (nb 06)        |           (nb 07)    BetterTogether
                                  |                            |                      (this nb)
                                  +--- stack Ensemble / KNNFewShot / inference-time compute
                                       on top of any of the above when the economics fit
```

The same guide as a table:

| Optimizer | Tunes | Needs | Relative cost | Reach for it when |
|---|---|---|---|---|
| `LabeledFewShot` | demos | a few labeled examples | free | you need a baseline in 30 seconds |
| `BootstrapFewShot` | demos | metric + ~20 examples | $ | first real optimizer on any new task |
| `BootstrapFewShotWithRandomSearch` | demos | same, + patience | $$ | squeeze more from the demo family |
| `KNNFewShot` | demos (per input) | trainset + embedder | $ + runtime lookup | inputs cluster into distinct regimes |
| `COPRO` | instructions | metric | $$ | want 0-shot prompts; hill-climb is enough |
| `MIPROv2` | instructions + demos | metric, 50+ examples | $$–$$$ | the default serious choice |
| `SIMBA` | instructions + demos | metric | $$$ | plateaued elsewhere; introspective rules help |
| `GEPA` | instructions | 5-arg feedback metric + strong reflection LM | $$$ | your metric can *explain* failures in text |
| `BootstrapFinetune` | LM weights | trainable model | $$$ + training | latency/cost pressure; narrow task |
| `BetterTogether` | prompts + weights | both of the above | $$$$ | you want the last few points and own the stack |
| `Ensemble` | program set | several compiled programs | runtime ×N | variance is your enemy; budget per call is not |

Not covered in this course (pointers): streaming & async serving (`dspy.streamify` / `asyncify`), `dspy.Parallel` for batch fan-out, and using GEPA as a pure inference-time searcher (notebook 07's sidebar). Observability — *watching* all of this run — is covered: it gets its own module in notebook 11.

## Recap

| Family | One-liner |
|---|---|
| demos (05) | bootstrap + filter is the workhorse; RS adds selection; KNN makes it per-input |
| instructions (06) | COPRO hill-climbs; MIPROv2 proposes grounded + searches with a surrogate |
| feedback (07) | GEPA turns metric text into reflective evolution |
| weights (09) | BootstrapFinetune moves traces into weights; BetterTogether alternates p ↔ w |
| structure (09) | Ensemble votes over diverse programs |
| chooser | data size → budget → feedback availability → weight access |

## Exercises

1. Three scenarios — pick the optimizer and defend it: (a) 15 labeled support tickets, cheap model, need something *tomorrow*; (b) 5,000 labeled examples, a rubric-heavy legal-summarization task, latency budget of one short prompt; (c) a pipeline whose metric already computes *which* stage failed and why, in prose. <details><summary>One reasonable answer set</summary>(a) BootstrapFewShot — data-light, robust, hours not days. (b) MIPROv2 medium/heavy aiming at 0-shot instructions (short prompts!), consider BootstrapFinetune if you own the model. (c) GEPA — that metric is already a feedback function; wire it into the 5-arg contract.</details>
2. Run the ensemble cell with `k in (1, 2, 4, 8, 16)` (five programs). Does the vote keep improving, or do the weak members drag it down? Relate to bagging intuition.
3. Sketch (prose, no code) a budgeted plan for a 500-example task: which optimizer first, what artifact it hands the next one, and where you would stop. There is a defensible answer in the BetterTogether strategy-string grammar.

**Next up:** the capstone — everything in this course, end to end, on real HotPotQA data. Same program shape, same metric discipline, same optimizers; only the world stops being fictional.